In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import os
from datetime import datetime

from metpy.calc import wind_direction, wind_speed, wind_components
from metpy.units import units
from ICONProcessor import ICONDataGrid

## Script settings

In [2]:
icon_file_30min = '/Users/geoalxx/Python/glacier_space/data_icon_h3/levante_v370_2030/v370_2030_LES_30min_51m_ml_35704.nc'  # data
icon_file = '/Users/geoalxx/Python/glacier_space/data_icon_h3/levante_v370_2030/v370_2030_LES_51m_ml_35704.nc'  # z_mc, z_ifc
icon_nlev = 30

wr_dir = '/Users/geoalxx/HEFEX III/WindRanger/averaged/'
cnt_height = 13

data_out_dir = '../data_out/'

## Load and prepare WindRanger data

In [3]:
wr_variables_ts = ['alt','lat','lon', 'heading', 'pitch', 'roll']
wr_variables_2D = ['U','V','W','DIR','VEL','SU','SV','SW','SNR','DQ']

In [4]:
wr_files = sorted(os.listdir(wr_dir))

# prepare data containers for time and time series variables
dim_h_wr = None
dim_time_wr = []
wr_df_ts = pd.DataFrame()

# prepare empty df for each 2D variable
wr_df_2D = {}
for v in wr_variables_2D:
    wr_df_2D[v] = pd.DataFrame()

for file in wr_files:
    if file.endswith('_averaged.nc'):
        # get time and height values
        ds = xr.open_dataset(wr_dir + file)
        time = pd.to_datetime(ds['time'].values, utc=True)
        height = ds['height'].values
        print('Processing', file,'- Height:', len(height), '- Time:', len(time))

        # do some quality checks
        if len(height) != cnt_height:
            print(f'Height dimension of {file} does not meet requirement. -> skipped!')
            continue
        elif dim_h_wr is None:
            dim_h_wr = height
            wr_file_ref = wr_dir + file
        elif not np.array_equal(height, dim_h_wr):
            print(f'Height dimension of {file} do not match. -> skipped!')
            continue

        # collect time series data
        dim_time_wr.extend(time)
        df_ts_tmp = pd.DataFrame(index=time)
        for v in wr_variables_ts:
            df_ts_tmp[v] = ds[v].values
        wr_df_ts = pd.concat([wr_df_ts, df_ts_tmp])

        # collect 2D data
        for v in wr_variables_2D:
            wr_df_2D[v] = pd.concat([wr_df_2D[v], pd.DataFrame(ds[v].values, index=time)])

Processing 20250806_000000_averaged.nc - Height: 7 - Time: 23
Height dimension of 20250806_000000_averaged.nc does not meet requirement. -> skipped!
Processing 20250806_150000_averaged.nc - Height: 13 - Time: 53
Processing 20250807_000000_averaged.nc - Height: 13 - Time: 78
Processing 20250807_125000_averaged.nc - Height: 13 - Time: 66
Processing 20250808_000000_averaged.nc - Height: 13 - Time: 144
Processing 20250809_000000_averaged.nc - Height: 13 - Time: 144
Processing 20250810_000000_averaged.nc - Height: 13 - Time: 144
Processing 20250811_000000_averaged.nc - Height: 13 - Time: 144
Processing 20250812_000000_averaged.nc - Height: 13 - Time: 144
Processing 20250813_000000_averaged.nc - Height: 13 - Time: 144
Processing 20250814_000000_averaged.nc - Height: 13 - Time: 144
Processing 20250815_000000_averaged.nc - Height: 13 - Time: 144
Processing 20250816_000000_averaged.nc - Height: 13 - Time: 144
Processing 20250817_000000_averaged.nc - Height: 13 - Time: 144
Processing 20250818_00

## Load and prepare ICON data

In [5]:
ds_30min = xr.open_dataset(icon_file_30min)
ds = xr.open_dataset(icon_file)

# dimension: time, height (full level and half level)
dim_time_i = pd.Series([ICONDataGrid.convert_float_dt(f) for f in ds_30min.time.values])
elev = ds.z_ifc.values[-1, 0]
dim_h_mc = ds.z_mc.values[-icon_nlev:, 0] - elev
dim_h_ifc = ds.z_ifc.values[-icon_nlev-1:, 0] - elev

# get data
icon_df_u = pd.DataFrame(ds_30min.u.values[:, -icon_nlev:, 0], index=dim_time_i, columns=dim_h_mc)
icon_df_v = pd.DataFrame(ds_30min.v.values[:, -icon_nlev:, 0], index=dim_time_i, columns=dim_h_mc)
icon_df_w = pd.DataFrame(ds_30min.v.values[:, -icon_nlev - 1:, 0], index=dim_time_i, columns=dim_h_ifc)

# calc wind direction and speed
icon_df_wspd = pd.DataFrame(wind_speed(icon_df_u.values * units('m/s'), icon_df_v.values * units('m/s')), index=dim_time_i, columns=dim_h_mc)
icon_df_wdir = pd.DataFrame(wind_direction(icon_df_u.values * units('m/s'), icon_df_v.values * units('m/s')), index=dim_time_i, columns=dim_h_mc)

## Build dataset

In [6]:
def get_var_attrs_ref(ref_ds_file, ref_var):
    # get reference dataset for meta data
    ref_ds = xr.open_dataset(ref_ds_file)
    attrs = {
        "long_name": ref_ds[ref_var].long_name,
        "standard_name": ref_ds[ref_var].standard_name,
        "units": ref_ds[ref_var].units
    }
    return attrs

def create_data_array(df, name, time_dim, height_dim, height_name, var_attrs):
    # get time from dataset
    time_naive = [dt.replace(tzinfo=None) for dt in df.index.to_numpy()]

    # create DataArray
    xr_data = xr.DataArray(
        df.values,
        coords={
            time_dim: (time_dim, time_naive, {"long_name": "Timestamp in UTC"}),
            height_dim: (height_dim, df.columns.to_numpy(), {"long_name": height_name}),
        },
        dims=[time_dim, height_dim],
        name=name,
        attrs=var_attrs
    )
    return xr_data

In [23]:
# Combine DataArrays

# add ICON data
ds_out = xr.Dataset({
    'icon_u'    : create_data_array(icon_df_u, 'icon_u', 'time_icon', 'height_icon_mc', 'height at full level center', get_var_attrs_ref(icon_file, 'u')),
    'icon_v'    : create_data_array(icon_df_v, 'icon_v', 'time_icon', 'height_icon_mc', 'height at full level center', get_var_attrs_ref(icon_file, 'v')),
    'icon_w'    : create_data_array(icon_df_w, 'icon_w', 'time_icon', 'height_icon_ifc', 'height at half level center', get_var_attrs_ref(icon_file, 'w')),
    'icon_wspd' : create_data_array(icon_df_wspd, 'icon_wspd', 'time_icon', 'height_icon_mc', 'ICON height at full level center', get_var_attrs_ref(wr_file_ref, 'VEL')),
    'icon_wdir' : create_data_array(icon_df_wdir, 'icon_wdir', 'time_icon', 'height_icon_mc', 'ICON height at full level center', get_var_attrs_ref(wr_file_ref, 'DIR')),
})

# add WindRanger data
for v in wr_variables_2D:
    if v == 'DIR':
        var_name = 'wr_wdir'
    elif v == 'VEL':
        var_name = 'wr_wspd'
    else:
        var_name = str.lower(f'wr_{v}')
    ds_out[var_name] = create_data_array(wr_df_2D[v], var_name, 'time_wr', 'height_wr', 'WindRanger height', get_var_attrs_ref(wr_file_ref, v))

# ToDo: sort and remove duplicates

# ToDo: add meta data (lat, lon, ...)

# Add global attributes
ds_out.attrs = {
    "title": f"Combined dataset of WindRanger data and corresponding ICON-LES data",
    "institution": "Humboldt-Universität zu Berlin",
    "source": "icon-2024.10 (https://gitlab.dkrz.de/icon/icon-model.git) and output of WindRanger measurements",
    "history": f"Created {datetime.now().strftime('%Y-%m-%d')}",
    "contact": "alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-9465-6761)",
    "experiment_id": "hefex3/v370_2030/exp_R3B15_51m",
    "campaign": "HEFEX III",
    "location": f"WindRanger (lon, lat, elev): {wr_df_ts['lon'].mean():.4f}°E, {wr_df_ts['lat'].mean():.4f}°N, {wr_df_ts['alt'].mean():.2f}m",
    #"StartTime": format_datetime64(ds_out.time.values[0]),
    #"StopTime": format_datetime64(ds_out.time.values[-1]),
    "comment": f"ICON simulation based on glacier outlines from OGGM SSP370, year 2030"
}

# Write
output_file = data_out_dir + f'H3_ICON_WindRanger_2D.nc'
ds_out.to_netcdf(output_file)
print(f'Output file saved as {output_file}.')

Output file saved as ../data_out/H3_ICON_WindRanger_2D.nc.


In [24]:
ds_out


<xarray.Dataset> Size: 3MB
Dimensions:          (time_icon: 1525, height_icon_mc: 30, height_icon_ifc: 31,
                      time_wr: 3419, height_wr: 13)
Coordinates:
  * time_icon        (time_icon) datetime64[ns] 12kB 2025-08-05 ... 2025-09-0...
  * height_icon_mc   (height_icon_mc) float32 120B 558.7 533.5 ... 15.0 5.0
  * height_icon_ifc  (height_icon_ifc) float32 124B 571.4 546.0 ... 10.0 0.0
  * time_wr          (time_wr) datetime64[ns] 27kB 2025-08-06T15:10:00 ... 20...
  * height_wr        (height_wr) int64 104B 0 1 2 3 4 5 6 7 8 9 10 11 12
Data variables: (12/15)
    icon_u           (time_icon, height_icon_mc) float32 183kB 3.401 ... 0.3577
    icon_v           (time_icon, height_icon_mc) float32 183kB -9.003 ... -3.003
    icon_w           (time_icon, height_icon_ifc) float32 189kB -9.367 ... -3...
    icon_wspd        (time_icon, height_icon_mc) float32 183kB 9.624 ... 3.024
    icon_wdir        (time_icon, height_icon_mc) float32 183kB 339.3 ... 353.2
    wr_u             (time_wr, height_wr) float32 178kB 1.548 1.649 ... nan nan
    ...               ...
    wr_wspd          (time_wr, height_wr) float32 178kB 3.152 2.932 ... nan nan
    wr_su            (time_wr, height_wr) float32 178kB 0.4945 0.5356 ... nan
    wr_sv            (time_wr, height_wr) float32 178kB 0.5454 0.4294 ... nan
    wr_sw            (time_wr, height_wr) float32 178kB 0.2735 0.3484 ... nan
    wr_snr           (time_wr, height_wr) float64 356kB -20.41 -20.07 ... -33.81
    wr_dq            (time_wr, height_wr) float64 356kB 1.0 1.0 1.0 ... 0.0 0.0
Attributes:
    title:          Combined dataset of WindRanger data and corresponding ICO...
    institution:    Humboldt-Universität zu Berlin
    source:         icon-2024.10 (https://gitlab.dkrz.de/icon/icon-model.git)...
    history:        Created 2026-04-13
    contact:        alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-946...
    experiment_id:  hefex3/v370_2030/exp_R3B15_51m
    campaign:       HEFEX III
    location:       WindRanger (lon, lat, elev): 10.7717°E, 46.8016°N, 2720.82m
    comment:        ICON simulation based on glacier outlines from OGGM SSP37...